# **KIỂM TRA DATASET DEEPFASHION**

In [1]:
import os
from pathlib import Path
import pandas as pd
from PIL import Image

# Cấu hình đường dẫn
root_dir = Path(r"C:\Users\Nguyen Ho Vinh  Hien\Downloads\DeepFashion")
img_dir = root_dir / "selected_images"

# Nếu tập dữ liệu khác tên file annotation thì bổ sung ở đây
annotation_candidates = [
    root_dir / "labels_front.csv",
    root_dir / "annotations.csv",
    root_dir / "deepfashion_multimodal.csv",
    root_dir / "annotation.csv",
    root_dir / "labels.csv",
]

annotation_path = next((p for p in annotation_candidates if p.exists()), None)
output_clean = root_dir / "deepfashion_multimodal_clean.csv"
log_path = root_dir / "deepfashion_validation_log.txt"

log_lines = []

def add_log(msg):
    print(msg)
    log_lines.append(str(msg))

# 2. Kiểm tra tồn tại file
add_log("KIỂM TRA DATASET DEEPFASHION-MULTIMODAL")
if annotation_path is None:
    add_log(f"[WARN] Không tìm thấy file annotation trong thư mục: {root_dir}")
    add_log("Danh sách file gợi ý: " + ", ".join(str(p.name) for p in annotation_candidates))
else:
    add_log(f"Tìm thấy file annotation: {annotation_path}")

if img_dir.exists():
    add_log(f"Tìm thấy thư mục ảnh: {img_dir}")
else:
    add_log(f"Không tìm thấy thư mục ảnh: {img_dir}")

# 3. Đọc & validate annotation
if annotation_path is not None:
    df = pd.read_csv(annotation_path)
    add_log(f"Tổng số dòng (samples): {len(df)}")
    add_log(f"Tổng số cột: {df.shape[1]}")
    add_log(f"Tổng số missing values: {int(df.isnull().sum().sum())}")

    add_log("\nTHÔNG TIN DATAFRAME")
    df.info()
    add_log("\n5 DÒNG ĐẦU TIÊN")
    print(df.head())
    add_log("\nDESCRIBE")
    print(df.describe(include='all').T)

    # Kiểm tra cột ảnh / ID
    image_like_cols = []
    for col in df.columns:
        c = col.lower()
        if any(k in c for k in ['image', 'img', 'file', 'filename', 'path', 'photo', 'id']):
            image_like_cols.append(col)

    add_log(f"\nCác cột có khả năng chứa thông tin ảnh/ID: {image_like_cols}")

    # Missing values theo cột
    null_summary = df.isnull().sum()
    null_cols = null_summary[null_summary > 0]
    add_log("\n=== NULL THEO CỘT ===")
    if null_cols.empty:
        add_log("Không có giá trị null.")
    else:
        print(null_cols)

    # Duplicate rows
    duplicate_count = int(df.duplicated().sum())
    add_log(f"\nSố dòng duplicate: {duplicate_count}")
    if duplicate_count > 0:
        df = df.drop_duplicates().copy()
        add_log("Đã xoá duplicate.")

    # Chuẩn hóa text columns
    object_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in object_cols:
        df[col] = df[col].astype(str).str.strip()

    # Xử lý missing values theo kiểu dữ liệu
    for col in df.columns:
        if df[col].isnull().sum() == 0:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            median_value = df[col].median()
            df[col] = df[col].fillna(median_value)
            add_log(f"Cột '{col}' numeric: điền bằng median = {median_value}")
        else:
            mode_value = df[col].mode(dropna=True)
            fill_value = mode_value.iloc[0] if not mode_value.empty else 'Unknown'
            df[col] = df[col].fillna(fill_value)
            add_log(f"Cột '{col}' category/text: điền bằng mode = {fill_value}")

    # Kiểm tra toàn vẹn ảnh - annotation
    if img_dir.exists():
        add_log("\n=== KIỂM TRA MAPPING ẢNH - ANNOTATION ===")
        image_files = {
            p.name.lower() for p in img_dir.iterdir()
            if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}
        }

        anno_image_names = set()
        for col in image_like_cols:
            anno_image_names.update({str(x).lower() for x in df[col].dropna() if str(x).strip() != ''})

        if anno_image_names:
            missing_in_img = sorted(anno_image_names - image_files)
            missing_in_anno = sorted(image_files - anno_image_names)
            add_log(f"Số annotation không có file ảnh thật: {len(missing_in_img)}")
            if missing_in_img:
                print(missing_in_img[:10])

            add_log(f"Số ảnh trên ổ đĩa không có trong annotation: {len(missing_in_anno)}")
            if missing_in_anno:
                print(missing_in_anno[:10])
        else:
            add_log("Không xác định được cột chứa tên ảnh / ID ở annotation.")

    # Kiểm tra ảnh bị hỏng / corrupt
    if img_dir.exists():
        add_log("\nKIỂM TRA TÍNH TOÀN VẸN CỦA ẢNH")
        valid_count = 0
        corrupt_count = 0
        corrupt_files = []

        for file in sorted(img_dir.iterdir()):
            if not file.is_file():
                continue
            if file.suffix.lower() not in {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}:
                continue

            try:
                with Image.open(file) as img:
                    img.verify()
                    img.load()
                valid_count += 1
            except Exception as e:
                corrupt_count += 1
                corrupt_files.append((file.name, str(e)))

        add_log(f"Số ảnh hợp lệ: {valid_count}")
        add_log(f"Số ảnh lỗi / corrupt: {corrupt_count}")
        if corrupt_files:
            add_log("Ví dụ ảnh lỗi:")
            for item in corrupt_files[:10]:
                print(item)

    # Lưu dữ liệu sạch
    df_clean = df.copy()
    output_clean.parent.mkdir(parents=True, exist_ok=True)
    df_clean.to_csv(output_clean, index=False)
    add_log(f"\nĐã lưu file clean: {output_clean}")

    # Ghi log
    with open(log_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(log_lines))
    add_log(f"Đã lưu log kiểm tra: {log_path}")

    add_log("\nKIỂM TRA KẾT THÚC")
    add_log("Data đã được chuẩn hóa sơ bộ và lưu ở phiên bản clean.")

else:
    add_log("\nERROR Không tìm thấy file annotation. Vui lòng kiểm tra lại đường dẫn dữ liệu.")
    add_log(f"Thư mục hiện tại: {root_dir}")
    add_log("Nếu dữ liệu nằm ở thư mục khác, cập nhật lại biến root_dir hoặc annotation_candidates.")

KIỂM TRA DATASET DEEPFASHION-MULTIMODAL
Tìm thấy file annotation: C:\Users\Nguyen Ho Vinh  Hien\Downloads\DeepFashion\labels_front.csv
Tìm thấy thư mục ảnh: C:\Users\Nguyen Ho Vinh  Hien\Downloads\DeepFashion\selected_images
Tổng số dòng (samples): 12278
Tổng số cột: 7
Tổng số missing values: 0

THÔNG TIN DATAFRAME
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12278 entries, 0 to 12277
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   image_id      12278 non-null  object
 1   caption       12278 non-null  object
 2   path          12278 non-null  object
 3   gender        12278 non-null  object
 4   product_type  12278 non-null  object
 5   product_id    12278 non-null  object
 6   image_type    12278 non-null  object
dtypes: object(7)
memory usage: 671.6+ KB

5 DÒNG ĐẦU TIÊN
                           image_id  \
0  MEN-Denim-id_00000089-28_1_front   
1  MEN-Denim-id_00000265-01_1_front   
2  MEN-Denim-id_00000